# 256 × 256 tile evaluation with full-frame reconstruction

Set **DATASET** and run all cells. Tile scores are read through the v1 cache.
For each method and image resolution, the tiles with lower mean Chamfer get
first choice of shared pixels. Each full-frame score is recomputed against the
original-size ground truth from the stitched event stream.

The default ranks and evaluates using the selected sequences, so this is a
retrospective comparison. For held-out evaluation, set **RANK_SEQUENCES** and
**SEQUENCES** to disjoint lists.

In [ ]:
from pathlib import Path

DATASET = "eds"
SPLIT = "test"
SEQUENCES = None          # None = all discovered sequences, excluding explicit RANK_SEQUENCES
RANK_SEQUENCES = None     # None = use the same sequences for ranking and evaluation
METHODS = None           # None = all methods with saved tile predictions
DATA_ROOT = None
PRED_ROOT = None
V1_REPORT_ROOT = None     # None = use the v1 report and cache location
V2_REPORT_ROOT = None     # None = reports/experiments/<dataset>_256/<split>/watch_img_size_v2/runs
TABLE_METRICS = None      # None = all 16 metrics; Chamfer is always included for ranking
VOXEL_BIN_MS = 10.0
MATCH_TILES = True        # match valid sequence/tile pairs between methods
MATCH_SEQUENCES = True    # match valid full-frame sequences between methods
SCORE_WORKERS = 1
CHAMFER_THREADS = 4

SHOW_BEST_TILE_PLOTS = False  # enable the v1 event plots for the lowest valid individual Chamfer
PLOT_METHOD = None
PLOT_SEQUENCE = None
PLOT_TILE = None
PLOT_START_MS = None
PLOT_WINDOW_MS = 40.0
RATE_BIN_MS = 10.0
MAX_PLOT_POINTS = 15_000

In [ ]:
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "experiments" / "score_stitched_image.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the Evtween repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks import event_rate_views_v2 as rates
from experiments import score_image_tiles as scoring
from experiments import score_stitched_image as stitched

metrics = tuple(dict.fromkeys(("chamfer_full", *(TABLE_METRICS or scoring.METRICS))))
requested = None
if SEQUENCES is not None:
    requested = list(dict.fromkeys([*SEQUENCES, *(RANK_SEQUENCES or [])]))
records, methods, tile_report = scoring.evaluate(
    DATASET, SPLIT, methods=METHODS, sequences=requested, metrics=metrics,
    data_root=DATA_ROOT, pred_root=PRED_ROOT, report_root=V1_REPORT_ROOT,
    voxel_bin_ms=VOXEL_BIN_MS, workers=SCORE_WORKERS, chamfer_threads=CHAMFER_THREADS,
)
raw = pd.DataFrame(records)
all_names = set(raw.sequence)
evaluation = set(SEQUENCES) if SEQUENCES is not None else all_names - set(RANK_SEQUENCES or [])
ranking_names = set(RANK_SEQUENCES) if RANK_SEQUENCES is not None else evaluation
if not evaluation or not ranking_names or not ranking_names <= all_names:
    raise ValueError("Choose nonempty evaluation and ranking sets with available sequences.")
tile_rows = raw[raw.sequence.isin(evaluation)]
print("Tile report", tile_report)
print("Ranking sequences", ", ".join(sorted(ranking_names)))
print("Evaluation sequences", ", ".join(sorted(evaluation)))
if ranking_names & evaluation:
    print("Retrospective ranking: ranking and evaluation share sequences.")


def valid_scores(rows, pair_columns, match):
    values = rows.assign(value=pd.to_numeric(rows.value, errors="coerce"))
    values = values[values.status.isin(scoring.VALID_STATUS) & np.isfinite(values.value)]
    if not match or values.empty:
        return values
    available_methods = values.groupby("metric")["method"].transform("nunique")
    matched_methods = values.groupby(["metric", *pair_columns])["method"].transform("nunique")
    return values[available_methods == matched_methods]


def display_table(table):
    display(HTML('<div style="overflow-x:auto">' + table.to_html(
        float_format=lambda value: f"{value:.5g}", na_rep="—") + "</div>"))

## Tile scores by method and position

Averages use all valid evaluation sequences at the given resolution and tile
position. This table contains the original tile scores before ownership is
applied. Coverage counts accompany each mean.

In [ ]:
tile_groups = ["source_width", "source_height", "tile", "method"]
valid = valid_scores(tile_rows, ["sequence", "tile"], MATCH_TILES)
grouped = valid.groupby([*tile_groups, "metric"])["value"]
tile_index = pd.MultiIndex.from_frame(
    tile_rows[tile_groups].drop_duplicates().sort_values(tile_groups))
tile_means = grouped.mean().unstack("metric").reindex(index=tile_index, columns=metrics)
tile_coverage = (grouped.size().unstack("metric").reindex(index=tile_index, columns=metrics)
                 .fillna(0).astype(int))
display_table(tile_means)
print("Coverage · evaluation sequences per tile mean")
display(tile_coverage)

## Full-frame scores after removing overlap

Each pixel is assigned to one tile. Priorities are derived separately for each
method and resolution using mean `chamfer_full` from the ranking sequences.
Tiles with no valid ranking score come last, with row-major tile ID as the
tie-breaker. The same ownership map is applied to every evaluation sequence
with that resolution. Each table entry is the arithmetic mean of the valid
full-frame sequence scores, with matching enabled by default.

In [ ]:
geometry = ["method", "source_width", "source_height"]
calibration = raw[(raw.metric == "chamfer_full") & raw.sequence.isin(ranking_names)]
rank_scores = valid_scores(calibration, ["sequence", "tile"], MATCH_TILES)
rank_rows = (rank_scores.groupby([*geometry, "tile"], as_index=False)
             .agg(mean_chamfer=("value", "mean"), n_rank=("value", "size")))
rankings = {key: group.set_index("tile").mean_chamfer.to_dict()
            for key, group in rank_rows.groupby(geometry)}

base_report = V2_REPORT_ROOT or (ROOT / "reports" / "experiments" /
                                f"{DATASET}_{scoring.TILE_SIZE}" / SPLIT / "watch_img_size_v2" / "runs")
full_records, report = stitched.evaluate(
    tile_rows.to_dict("records"), rankings, dataset=DATASET, split=SPLIT, metrics=metrics,
    report_root=base_report, tile_report=tile_report, rank_sequences=ranking_names,
    voxel_bin_ms=VOXEL_BIN_MS, workers=SCORE_WORKERS, chamfer_threads=CHAMFER_THREADS,
)
full_rows = pd.DataFrame(full_records)
full_valid = valid_scores(full_rows, ["sequence"], MATCH_SEQUENCES)
groups = full_valid.groupby(["method", "metric"])["value"]
index = pd.Index(methods, name="method")
full_means = groups.mean().unstack("metric").reindex(index=index, columns=metrics)
full_coverage = (groups.size().unstack("metric").reindex(index=index, columns=metrics)
                 .fillna(0).astype(int))
tile_means.to_csv(report / "tile_method_means.csv")
tile_coverage.to_csv(report / "tile_method_coverage.csv")
full_means.to_csv(report / "stitched_method_means.csv")
full_coverage.to_csv(report / "stitched_method_coverage.csv")
print("V2 report", report)
display_table(full_means)
print("Coverage · full-frame sequences per mean")
display(full_coverage)
unavailable = full_rows[full_rows.status.isin(["missing_prediction", "error"])]
if not unavailable.empty:
    display(unavailable[["sequence", "method", "metric", "status", "error"]].head(10))
    print(f"{len(unavailable)} unavailable full-frame scores. Details in stitched_results.csv.")

## Ownership map in original pixel coordinates

Solid colours show which tile provides events for each pixel. The dashed
outlines show the original overlapping crops. Hover for the owning tile, the
number of input tiles that covered the pixel, and the owner's ranking score.

In [ ]:
def show_and_save(figure, name):
    path = report / f"{name}.html"
    config = {"displaylogo": False, "responsive": True}
    figure.write_html(str(path), include_plotlyjs=True, config=config)
    figure.show(renderer="plotly_mimetype+notebook", config=config)
    print(path)


fig = go.Figure()
panels, allocations = [], []
palette = ["#bdd7ee", "#8cd3c5", "#f5c68c", "#beaee2", "#f3aeb4", "#bcd78e", "#e5ce9a", "#a6cee3"]
for method, width, height in tile_rows[geometry].drop_duplicates().itertuples(index=False, name=None):
    means = rankings.get((method, width, height), {})
    order, owner, coverage = stitched.ownership(width, height, means)
    xs = sorted({edge for tile in order for edge in (tile.x, tile.x + scoring.TILE_SIZE)})
    ys = sorted({edge for tile in order for edge in (tile.y, tile.y + scoring.TILE_SIZE)})
    labels = {tile.index: tile.name for tile in order}
    z = owner[np.ix_(ys[:-1], xs[:-1])]
    overlap = coverage[np.ix_(ys[:-1], xs[:-1])]
    text = []
    for owners, covers in zip(z, overlap):
        line = []
        for index, count in zip(owners, covers):
            label = labels[index]
            mean = f"{means[label]:.3f}" if label in means else "unranked"
            line.append(f"{label}<br>Input tiles covering this region: {count}<br>Mean Chamfer: {mean}")
        text.append(line)
    scale = [[bound / len(order), palette[i % len(palette)]]
             for i in range(len(order)) for bound in (i, i + 1)]
    visible = not panels
    fig.add_heatmap(x=xs, y=ys, z=z, zmin=-.5, zmax=len(order)-.5, colorscale=scale,
                    showscale=False, text=text, hovertemplate="%{text}<extra></extra>", visible=visible)
    outline_x, outline_y = [], []
    for rank, tile in enumerate(order, 1):
        x, y, w, h = tile.crop
        outline_x.extend([x, x+w, x+w, x, x, None])
        outline_y.extend([y, y, y+h, y+h, y, None])
        pixels = int(np.count_nonzero(owner == tile.index))
        allocations.append(dict(method=method, source_width=width, source_height=height,
                                tile=tile.name, priority=rank, mean_chamfer=means.get(tile.name),
                                x0=x, y0=y, pixels_kept=pixels,
                                frame_percentage=100 * pixels / (width * height)))
    fig.add_scatter(x=outline_x, y=outline_y, mode="lines", line=dict(color="#34455c", dash="dot"),
                    showlegend=False, hoverinfo="skip", visible=visible)
    panels.append((f"{method} · {width} × {height}", width, height))

buttons = [dict(label=label, method="update", args=[
    {"visible": [j // 2 == i for j in range(len(fig.data))]},
    {"title.text": f"{DATASET} · {label}", "xaxis.range": [0, width], "yaxis.range": [height, 0]},
]) for i, (label, width, height) in enumerate(panels)]
label, width, height = panels[0]
fig.update_layout(template="plotly_white", title=f"{DATASET} · {label}", height=540,
                  updatemenus=[dict(buttons=buttons, x=0, y=1.12)])
fig.update_xaxes(title="x (px)", range=[0, width], constrain="domain", showgrid=False)
fig.update_yaxes(title="y (px)", range=[height, 0], scaleanchor="x", constrain="domain", showgrid=False)
show_and_save(fig, "tile_ownership_map")
priority = pd.DataFrame(allocations).merge(rank_rows[geometry + ["tile", "n_rank"]],
                                           on=geometry + ["tile"], how="left")
priority["n_rank"] = priority["n_rank"].fillna(0).astype(int)
priority.to_csv(report / "tile_ownership.csv", index=False)
display_table(priority)

## Optional plots for the best individual tile

Turn on **SHOW_BEST_TILE_PLOTS** to show and export the comparisons with
the matching cropped GT from the first notebook.

In [ ]:
choices = raw[(raw.metric == "chamfer_full") & raw.sequence.isin(evaluation)]
choices = valid_scores(choices, ["sequence", "tile"], match=False)
for column, selected in (("method", PLOT_METHOD), ("sequence", PLOT_SEQUENCE), ("tile", PLOT_TILE)):
    if selected is not None:
        choices = choices[choices[column] == selected]
best = choices.sort_values(["value", "method", "sequence", "tile"]).head(1)
if SHOW_BEST_TILE_PLOTS and not best.empty:
    display(best[["method", "sequence", "tile", "x0", "y0", "value"]].reset_index(drop=True))
else:
    best = best.iloc[:0]

In [ ]:
def plot_isi(streams, meta, title, tag):
    intervals = {name: np.sort(scoring.bench.m.isi_distribution(events, meta["width"], meta["duration"]))
                 for name, events in streams.items()}
    upper = max([values[-1] for values in intervals.values() if values.size] + [.001])
    grid = np.r_[0, np.geomspace(1e-6, upper, 250)]
    fig = go.Figure()
    colors = ("#252b34", "#2563eb")
    for (name, values), color in zip(intervals.items(), colors):
        if values.size:
            fig.add_scatter(x=np.log10(1 + grid / .001), y=np.searchsorted(values, grid, side="right") / values.size,
                            customdata=grid, name=name, mode="lines", line=dict(color=color, width=2),
                            hovertemplate="%{customdata:.4g} ms · CDF %{y:.3f}<extra>%{fullData.name}</extra>")
    ticks = np.asarray([v for v in (0, .001, .01, .1, 1, 10, 100, 1000, 10000, 100000) if v <= upper])
    fig.update_layout(template="plotly_white", title=f"{title} · ISI CDF", height=470,
                      xaxis=dict(title="ISI (ms)", tickvals=np.log10(1 + ticks / .001),
                                 ticktext=[f"{v:g}" for v in ticks]),
                      yaxis=dict(title="Cumulative fraction", range=[0, 1.02]))
    if not fig.data:
        fig.add_annotation(text="No inter-event intervals in either stream", showarrow=False,
                           x=.5, y=.5, xref="paper", yref="paper")
    show_and_save(fig, f"{tag}_isi_cdf")


def clip_for_plots(streams, duration):
    if not np.isfinite(PLOT_WINDOW_MS) or PLOT_WINDOW_MS <= 0:
        raise ValueError("PLOT_WINDOW_MS must be positive.")
    span = PLOT_WINDOW_MS / 1000
    gt = streams["Ground truth"]
    if PLOT_START_MS is None:
        edges = scoring.voxels.time_edges(0., duration, span * 1000)
        start = float(edges[np.diff(np.searchsorted(gt["t"], edges)).argmax()])
    else:
        start = PLOT_START_MS / 1000
    stop = min(start + span, duration)
    if not np.isfinite(start) or not 0 <= start < stop:
        raise ValueError("Choose a plot start inside the sequence.")
    cut = {}
    for name, events in streams.items():
        a, b = np.searchsorted(events["t"], [start, stop])
        cut[name] = {key: value[a:b] for key, value in events.items()}
    return cut, start, stop


def display_indices(mask):
    idx = np.flatnonzero(mask)
    if idx.size > MAX_PLOT_POINTS:
        idx = idx[np.linspace(0, idx.size - 1, MAX_PLOT_POINTS, dtype=int)]
    return idx


def plot_spacetime(streams, meta, start, stop, title, tag):
    gt = streams["Ground truth"]
    sy = int(np.argmax(np.bincount(gt["y"], minlength=meta["height"])))
    sx = int(np.argmax(np.bincount(gt["x"], minlength=meta["width"])))
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"x–t at y={sy}", f"y–t at x={sx}"))
    cloud = go.Figure()
    for (name, events), color in zip(streams.items(), ("#4b5563", "#2563eb")):
        for polarity, symbol, positive in (("ON", "circle", True), ("OFF", "x", False)):
            sign = (events["p"] > 0) == positive
            label = f"{name} · {polarity}"
            for col, (other, pos, axis) in enumerate((("y", sy, "x"), ("x", sx, "y")), 1):
                idx = display_indices(sign & (events[other] == pos))
                fig.add_scattergl(x=events[axis][idx], y=1000 * events["t"][idx], name=label,
                                  mode="markers", legendgroup=label, showlegend=col == 1,
                                  marker=dict(color=color, symbol=symbol, size=6, opacity=.7), row=1, col=col)
            idx = display_indices(sign)
            cloud.add_scatter3d(x=events["x"][idx], y=1000 * events["t"][idx], z=events["y"][idx],
                                name=label, mode="markers",
                                marker=dict(color=color, symbol=symbol, size=2.3, opacity=.55))
    window = f"{1000 * start:.1f}–{1000 * stop:.1f} ms"
    fig.update_layout(template="plotly_white", height=510, title=f"{title} · {window}")
    fig.update_xaxes(title="x (px)", range=[0, meta["width"] - 1], row=1, col=1)
    fig.update_xaxes(title="y (px)", range=[0, meta["height"] - 1], row=1, col=2)
    fig.update_yaxes(title="Time (ms)", range=[1000 * start, 1000 * stop])
    cloud.update_layout(template="plotly_white", height=620, title=f"{title} · {window} · 3D events",
                        scene=dict(xaxis_title="x (px)", yaxis_title="Time (ms)", zaxis_title="y (px)",
                                   zaxis=dict(autorange="reversed"), aspectmode="cube"))
    show_and_save(fig, f"{tag}_spacetime")
    show_and_save(cloud, f"{tag}_cloud3d")


if not best.empty:
    row = best.iloc[0]
    gt, pred, meta = scoring.load_example(row)
    streams = {"Ground truth": gt, row["method"]: pred}
    title = (f"{DATASET}/{row['sequence']}/{row['tile']} · {row['method']}"
             f" · Chamfer {row['value']:.4g}")
    tag = f"{row['sequence']}_{row['tile']}_{row['method']}"
    duration = float(meta["duration"])
    if not np.isfinite(RATE_BIN_MS) or RATE_BIN_MS <= 0 or MAX_PLOT_POINTS < 1:
        raise ValueError("Choose a positive rate-bin width and point cap.")
    rate_bin_ms = max(RATE_BIN_MS, np.ceil(duration * 1000 / 5000))
    rate_onoff, rate_total, _ = rates.make_rate_figures(streams, 0, duration, rate_bin_ms,
                                                        f"{title} · {rate_bin_ms:g} ms bins")
    show_and_save(rate_onoff, f"{tag}_event_rate_on_off")
    show_and_save(rate_total, f"{tag}_event_rate_total")
    plot_isi(streams, meta, title, tag)
    cut, start, stop = clip_for_plots(streams, duration)
    plot_spacetime(cut, meta, start, stop, title, tag)